# Circuit Composition

Build larger circuits from reusable subcircuits using the shared library.

**Objectives:**
- Compose circuits using `Circuit.add_circuit()`
- Build parameterized circuit functions for variational workflows
- Use the QFT from the library as a building block
- Understand circuit depth vs. width tradeoffs

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.
<!-- browser-runnable -->


In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt

from lib.circuits import bell_pair, ghz_state, qft_circuit
from lib.utils.results import parse_counts
from lib.utils.visualization import plot_histogram

device = LocalSimulator()

## 1. Composing Circuits with `add_circuit()`

The Braket SDK lets you append one circuit to another. This is how you build
larger algorithms from reusable sub-circuits.

In [ ]:
# Build a circuit by composing sub-circuits
# Step 1: Create a Bell pair on qubits 0,1
# Step 2: Add another Bell pair on qubits 2,3

main_circuit = Circuit()
main_circuit.add_circuit(bell_pair(qubit_0=0, qubit_1=1))
main_circuit.add_circuit(bell_pair(qubit_0=2, qubit_1=3))

print(main_circuit)
print(f"\nTotal qubits: {main_circuit.qubit_count}")
print(f"Circuit depth: {main_circuit.depth}")

# Run and verify: q0,q1 are correlated AND q2,q3 are correlated
result = device.run(main_circuit, shots=1000).result()
counts = result.measurement_counts

print(f"\nSample results: {dict(list(counts.items())[:6])}")
for bitstring in counts:
    assert bitstring[0] == bitstring[1], f"q0/q1 disagree: {bitstring}"
    assert bitstring[2] == bitstring[3], f"q2/q3 disagree: {bitstring}"
print("Verified: q0==q1 and q2==q3 in every measurement (two independent Bell pairs).")

## 2. Using QFT as a Building Block

The Quantum Fourier Transform is a key subroutine in many algorithms (QPE, Shor's, etc.).
Our library provides `qft_circuit(n_qubits)` ready to compose into larger circuits.

In [ ]:
# Examine the QFT circuit structure
qft3 = qft_circuit(n_qubits=3)
print("3-qubit QFT circuit:")
print(qft3)
print(f"\nDepth: {qft3.depth}, Qubit count: {qft3.qubit_count}")

In [ ]:
# QFT on |000> should produce uniform distribution over all 8 states
result = device.run(qft3, shots=8000).result()
counts = parse_counts(result)

fig = plot_histogram(counts, title="QFT|000> -- Should Be Approximately Uniform")
plt.show()

# Verify approximate uniformity
expected_per_state = 8000 / 8
for state, count in counts.items():
    deviation = abs(count - expected_per_state) / expected_per_state
    assert deviation < 0.15, f"State {state} deviates too much: {count}"
print(f"All 8 states within 15% of uniform ({expected_per_state:.0f} expected each).")

## 3. Parameterized Circuit Functions

For variational algorithms (VQE, QAOA, QML), you need circuits whose gates depend on
tunable parameters. Define these as Python functions that return a Circuit.

In [ ]:
def variational_layer(n_qubits, params):
    """A single variational layer: Ry rotations + entangling CNOTs."""
    circuit = Circuit()
    # Rotation sub-layer
    for i in range(n_qubits):
        circuit.ry(i, params[i])
    # Entangling sub-layer (linear chain)
    for i in range(n_qubits - 1):
        circuit.cnot(i, i + 1)
    return circuit


def variational_circuit(n_qubits, n_layers, all_params):
    """Stack multiple variational layers."""
    circuit = Circuit()
    for layer in range(n_layers):
        layer_params = all_params[layer * n_qubits:(layer + 1) * n_qubits]
        circuit.add_circuit(variational_layer(n_qubits, layer_params))
    return circuit


# Example: 3 qubits, 2 layers, random parameters
n_qubits, n_layers = 3, 2
params = np.random.uniform(0, 2 * np.pi, size=n_qubits * n_layers)

circuit = variational_circuit(n_qubits, n_layers, params)
print(f"Variational circuit ({n_qubits} qubits, {n_layers} layers):")
print(circuit)
print(f"\nDepth: {circuit.depth}, Parameters: {len(params)}")

In [ ]:
# Show how different parameters produce different output distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

param_sets = [
    np.zeros(6),                          # All zeros: circuit does nothing
    np.array([np.pi]*3 + [0]*3),          # First layer flips all qubits
    np.random.uniform(0, 2*np.pi, 6),     # Random params
]
titles = ['All params = 0', 'First layer = pi', 'Random params']

for ax, params, title in zip(axes, param_sets, titles):
    circuit = variational_circuit(3, 2, params)
    result = device.run(circuit, shots=2000).result()
    counts = result.measurement_counts
    
    all_states = [format(i, '03b') for i in range(8)]
    probs = [counts.get(s, 0) / 2000 for s in all_states]
    
    ax.bar(all_states, probs, color='#232f3e')
    ax.set_title(title)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Probability')

plt.suptitle('Same Circuit Structure, Different Parameters', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Circuit Metrics: Depth and Width

- **Width** = number of qubits
- **Depth** = number of time steps (layers of gates that can't run in parallel)

On real hardware, lower depth = less decoherence = better results.

In [ ]:
# Compare depth of different circuits
circuits = {
    "Bell pair": bell_pair(),
    "3-qubit GHZ": ghz_state(3),
    "5-qubit GHZ": ghz_state(5),
    "3-qubit QFT": qft_circuit(3),
    "4-qubit QFT": qft_circuit(4),
    "Variational (3q, 1L)": variational_circuit(3, 1, np.ones(3)),
    "Variational (3q, 3L)": variational_circuit(3, 3, np.ones(9)),
}

print(f"{'Circuit':<25} {'Qubits':<8} {'Depth':<8} {'Gates'}")
print("-" * 55)
for name, circ in circuits.items():
    n_gates = sum(1 for _ in circ.instructions)
    print(f"{name:<25} {circ.qubit_count:<8} {circ.depth:<8} {n_gates}")

## 5. Exercises

Two exercises to close out Foundations. Each has tiered hints -- expand only
what you need -- and a check cell that tells you when you have it.

### Exercise 1 — The 4-qubit QFT, verified uniform

Section 2 checked the 3-qubit QFT against a uniform distribution. Scale the
same experiment to 4 qubits: build the QFT with the shared library, run it
on |0000>, and confirm all 16 outcomes appear with roughly equal probability.

Define `qft4_counts`: the measurement counts from running your 4-qubit QFT
with at least 8000 shots.

<details><summary>Hint 1 — nudge</summary>

The QFT maps |00...0> to an equal superposition of every basis state. With 4
qubits that is 16 outcomes at probability 1/16 each — sampling noise shrinks
like $1/\sqrt{\text{shots}}$, which is why the prompt asks for so many shots.

</details>
<details><summary>Hint 2 — approach</summary>

Follow the Section 2 pattern: build with `qft_circuit(n_qubits=...)`, run it
with `device.run(..., shots=...)`, and convert with `parse_counts(...)`. A
`plot_histogram(...)` makes the uniformity easy to eyeball.

</details>

In [ ]:
# Exercise 1: Build a 4-qubit QFT and verify QFT|0000> is uniform over all 16 outcomes.
# Define: qft4_counts -- measurement counts from at least 8000 shots.

# TODO: your code here

In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    qft4_total = sum(qft4_counts.values())
    assert qft4_total >= 8000, "run at least 8000 shots for stable statistics"
    assert all(len(k) == 4 for k in qft4_counts), "the QFT should span 4 qubits"
    assert len(qft4_counts) == 16, "QFT|0000> should reach ALL 16 basis states"
    assert all(abs(v / qft4_total - 1 / 16) < 0.03 for v in qft4_counts.values()), (
        "every outcome's frequency should sit close to the uniform 1/16"
    )

### Exercise 2 — A reusable entanglement ring

Composition pays off when the building block is a function. Write
`entanglement_ring(n_qubits)` returning a Circuit that applies H to qubit 0,
then CNOTs from each qubit to the next around a ring: 0->1, 1->2, ...,
and finally n-1 back to 0.

Define `entanglement_ring` (the function, working for any qubit count) and
`ring4_counts`: the measurement counts from running `entanglement_ring(4)`
with at least 1000 shots. Look closely at which outcomes appear — the
wrap-around CNOT hides a surprise.

<details><summary>Hint 1 — nudge</summary>

The first n-1 CNOTs are exactly the GHZ chain you have already built. The
ring adds one final CNOT that closes the loop — trace by hand what that gate
does to each branch of the superposition.

</details>
<details><summary>Hint 2 — approach</summary>

Inside the function, start a fresh `Circuit`, apply `h(0)`, then loop `i`
over `range(n_qubits)` applying a `cnot` from `i` to its ring neighbor — the
modulo operator turns "the next qubit" into "wraps back to 0" on the last
step. Then run `entanglement_ring(4)` on the device and collect counts with
`parse_counts(...)`.

</details>

In [ ]:
# Exercise 2: Write entanglement_ring(n_qubits) and run it on 4 qubits.
# Define: entanglement_ring -- the circuit-building function, and
# ring4_counts -- measurement counts for entanglement_ring(4), at least 1000 shots.

# TODO: your code here

In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    ring6 = entanglement_ring(6)
    assert ring6.qubit_count == 6, "the function should work for any n, not just 4"
    assert len(ring6.instructions) == 7, (
        "an n-qubit ring is one H plus n CNOTs -- check your loop bounds"
    )
    ring4_total = sum(ring4_counts.values())
    assert ring4_total >= 1000, "run at least 1000 shots"
    assert all(len(k) == 4 for k in ring4_counts), "the 4-qubit ring measures 4 bits"
    assert len(ring4_counts) == 2, (
        "the ring should collapse to exactly TWO outcomes -- recheck the CNOT order"
    )
    assert all(k[1] == k[2] == k[3] for k in ring4_counts), (
        "qubits 1-3 should always agree, like a GHZ core"
    )
    assert all(k[0] == "0" for k in ring4_counts), (
        "trace the final wrap-around CNOT by hand: what does it leave on qubit 0?"
    )
    assert all(abs(v / ring4_total - 0.5) < 0.15 for v in ring4_counts.values()), (
        "the two outcomes should each appear roughly half the time"
    )

## Summary

- `Circuit.add_circuit()` composes sub-circuits into larger algorithms
- Library functions (`bell_pair`, `ghz_state`, `qft_circuit`) serve as building blocks
- Parameterized circuit functions enable variational algorithms (VQE, QAOA, QML)
- Circuit depth and width are key metrics for hardware feasibility
- Lower depth = less noise on real quantum hardware

**Congratulations!** You've completed the Foundations section. You now have the tools to:
- Build and run quantum circuits
- Create entangled states
- Analyze measurement statistics
- Compose complex circuits from reusable parts

**Next section:** [02-hardware](../../02-hardware/GUIDE.md) -- explore real quantum hardware devices on Amazon Braket.